# DG-2 structural walkthrough — basis-independence and the displaced bath

**Decision Gate:** DG-2 (structural sub-claims under the Council-cleared displacement-profile registry).  
**Verdict:** structural sub-claims PASS (2026-05-04); literal fourth-order $K_2$–$K_4$ recursion still pending.  
**Cards exercised:** [B3](../benchmarks/benchmark_cards/B3_cross-basis-structural-identity_v0.1.0.yaml), [B4](../benchmarks/benchmark_cards/B4-conv-registry_v0.1.0.yaml).  
**Authoritative status:** [`docs/validity_envelope.md`](../docs/validity_envelope.md).

This notebook demonstrates two of the four DG-2 structural-sub-claim families:

1. **Card B3** — basis-independence of `K_from_generator` at $d=2$ for a non-trivial (dissipative) generator. Expectation: $K$ in the matrix-unit basis equals $K$ in the $\mathfrak{su}(d)$-generator basis at machine precision.
2. **Card B4** — coherently-displaced bath under the Council-cleared displacement-profile registry. Expectation: $K(t)$ inherits a displacement-driven $\bar D_1\neq 0$ contribution that vanishes in the thermal limit ($\alpha_0\to 0$), recovering the DG-1 thermal A3 result.

> **Citation note (DG-2 ruling).** Citations of Entry 3.B.3 / 4.B.2 must name the registered displacement profile (one of `delta-omega_c`, `delta-omega_S`, `sqrt-J`, `gaussian`) under which the sub-claim is verified.

## 0. Setup

In [ ]:
import numpy as np

import cbg
from cbg import tcl_recursion as tr
from cbg.basis import matrix_unit_basis, su_d_generator_basis
from cbg.effective_hamiltonian import K_from_generator

print(f"cbg.__version__       = {cbg.__version__}")
print(f"cbg.__ledger_anchor__ = {cbg.__ledger_anchor__}")

## 1. Card B3 — basis-independence on a dissipative generator

Build a Hermiticity-preserving, trace-annihilating generator that is *not* purely unitary:

$$
L[X] = -i[H_S, X] + \gamma\,\bigl(\sigma_-\,X\,\sigma_+ - \tfrac{1}{2}\{\sigma_+\sigma_-, X\}\bigr).
$$

Letter Eq. (6) is basis-independent — computing $K$ in the matrix-unit basis $\{|j\rangle\langle k|\}$ and in the $\mathfrak{su}(2)$ generator basis $\{I,\sigma_x,\sigma_y,\sigma_z\}/\sqrt{2}$ must agree.

In [ ]:
sigma_z = np.array([[1, 0], [0, -1]], dtype=complex)
sigma_minus = np.array([[0, 0], [1, 0]], dtype=complex)
sigma_plus = sigma_minus.conj().T

H_S = 0.5 * sigma_z
gamma = 0.1

def L_dissipative(X):
    coh = -1j * (H_S @ X - X @ H_S)
    diss = gamma * (
        sigma_minus @ X @ sigma_plus
        - 0.5 * (sigma_plus @ sigma_minus @ X + X @ sigma_plus @ sigma_minus)
    )
    return coh + diss

K_mu = K_from_generator(L_dissipative, matrix_unit_basis(d=2))
K_su = K_from_generator(L_dissipative, su_d_generator_basis(d=2))

print("K_matrix-unit =\n", K_mu)
print("K_su(d) =\n", K_su)
print(f"||K_matrix-unit - K_su(d)||_F = {np.linalg.norm(K_mu - K_su):.2e}")

## 2. Card B4 — coherently-displaced bath, $\sigma_z$ coupling

Spec (one of four registered profiles in `B4-conv-registry_v0.1.0.yaml`): system $H_S=(\omega/2)\sigma_z$, coupling $A=\sigma_z$, Ohmic spectral density with $\alpha=0.05$, $\omega_c=10\omega$, *coherent-displaced* bath with profile `delta-omega_S` and displacement amplitude $\alpha_0=1.0$.

In [ ]:
t_grid = np.linspace(0.0, 5.0, 41)

spectral = {
    "family": "ohmic",
    "coupling_strength": 0.05,
    "cutoff_frequency": 10.0,
}

K_displaced = tr.K_total_displaced_on_grid(
    N_card=2,
    t_grid=t_grid,
    system_hamiltonian=H_S,
    coupling_operator=sigma_z,
    bath_state={
        "family": "coherent_displaced",
        "displacement_profile": "delta-omega_S",
        "parameters": {"alpha_0": 1.0, "omega_S": 1.0},
    },
    spectral_density=spectral,
)

print(f"K_displaced.shape = {K_displaced.shape}")
omega_r_disp = 2.0 * K_displaced[:, 0, 0].real
print(f"omega_r(t) under displacement (sample): "
      f"{omega_r_disp[0]:.4f}, {omega_r_disp[len(t_grid)//2]:.4f}, "
      f"{omega_r_disp[-1]:.4f}")

**Thermal-limit consistency.** Sending the displacement amplitude $\alpha_0 \to 0$ should reduce the displaced result to the DG-1 thermal result of Card A3.

In [ ]:
K_thermal = tr.K_total_thermal_on_grid(
    N_card=2,
    t_grid=t_grid,
    system_hamiltonian=H_S,
    coupling_operator=sigma_z,
    bath_state={"family": "thermal", "temperature": 0.0},
    spectral_density=spectral,
)

K_disp_zero = tr.K_total_displaced_on_grid(
    N_card=2,
    t_grid=t_grid,
    system_hamiltonian=H_S,
    coupling_operator=sigma_z,
    bath_state={
        "family": "coherent_displaced",
        "displacement_profile": "delta-omega_S",
        "parameters": {"alpha_0": 0.0, "omega_S": 1.0},
    },
    spectral_density=spectral,
)

diff = np.max(np.abs(K_disp_zero - K_thermal))
print(f"max |K_displaced(alpha_0=0) - K_thermal(T=0)| = {diff:.2e}")

## 3. Outcome

Both structural properties hold at machine precision:

- **Basis-independence** (B3): $K$ computed in the matrix-unit and $\mathfrak{su}(d)$-generator bases agrees to numerical zero on a non-trivial dissipative $L$.
- **Displaced-bath registry** (B4, profile `delta-omega_S`): $K(t)$ acquires a displacement-driven shift that vanishes continuously as $\alpha_0\to 0$, recovering the DG-1 thermal result.

These structural sub-claims authorise citation of CL-2026-005 v0.4 Entries 1.A, 1.B.3, 1.D, 3.B.3, and 4.B.2 within the registered displacement-profile envelope. The literal fourth-order $K_2$–$K_4$ recursion at perturbative_order $\geq 4$ remains pending; see [`docs/validity_envelope.md`](../docs/validity_envelope.md).